In [1]:
"""
Apple-to-apple KMeans benchmark:
  - sklearn KMeans (CPU)
  - cuML KMeans (GPU)
  - FlashKMeans (GPU)
 
Usage:
    pip install flash-kmeans scikit-learn
    python kmeans_benchmark.py
"""

import time
import torch
import numpy as np
 
# ── Config ────────────────────────────────────────────────────────────────────
N = 100_000   # number of samples
D = 128       # dimensionality
K = 1000      # number of clusters
SEED = 42
# ─────────────────────────────────────────────────────────────────────────────
 
np.random.seed(SEED)
torch.manual_seed(SEED)
 
# Shared data
data_np = np.random.randn(N, D).astype(np.float32)
data_torch = torch.tensor(data_np, device="cuda", dtype=torch.float32)
 
results = {}
 

In [2]:
# ── 1. sklearn KMeans (CPU) ───────────────────────────────────────────────────
print("Running sklearn KMeans (CPU)...")
from sklearn.cluster import KMeans as SklearnKMeans
 
t0 = time.perf_counter()
sk = SklearnKMeans(n_clusters=K, n_init=1, max_iter=100, random_state=SEED)
sk.fit(data_np)
t1 = time.perf_counter()
results["sklearn (CPU)"] = {
    "time_s": t1 - t0,
    "inertia": sk.inertia_,
}
print(f"  Done in {t1-t0:.2f}s  |  inertia={sk.inertia_:.4e}")

Running sklearn KMeans (CPU)...
  Done in 12.40s  |  inertia=1.1407e+07


In [3]:
# ── 2. cuML KMeans (GPU) ─────────────────────────────────────────────────────
print("Running cuML KMeans (GPU)...")
from cuml.cluster import KMeans as CumlKMeans
import cupy as cp
 
data_cp = cp.array(data_np)
# torch.cuda.synchronize()
t0 = time.perf_counter()
ck = CumlKMeans(n_clusters=K, n_init=1, max_iter=100, random_state=SEED)
ck.fit(data_cp)
# torch.cuda.synchronize()
t1 = time.perf_counter()
results["cuML (GPU)"] = {
    "time_s": t1 - t0,
    "inertia": float(ck.inertia_),
}
print(f"  Done in {t1-t0:.2f}s  |  inertia={float(ck.inertia_):.4e}")

Running cuML KMeans (GPU)...
  Done in 0.28s  |  inertia=1.1531e+07


In [4]:
# ── 3. FlashKMeans (GPU) ─────────────────────────────────────────────────────
print("Running FlashKMeans (GPU)...")
from flash_kmeans import FlashKMeans
 
t0 = time.perf_counter()
fk = FlashKMeans(d=D, k=K, niter=100, seed=SEED)
labels = fk.fit_predict(data_torch)
t1 = time.perf_counter()
 
# Compute inertia manually for fair comparison
centers = fk.centroids_b.squeeze(0)  # (K, D)
assigned_centers = centers[labels]  # (N, D)
dists = torch.sum((data_torch - assigned_centers) ** 2)
flash_inertia = dists.item()
 
results["FlashKMeans (GPU)"] = {
    "time_s": t1 - t0,
    "inertia": flash_inertia,
}
print(f"  Done in {t1-t0:.2f}s  |  inertia={flash_inertia:.4e}")

Running FlashKMeans (GPU)...
  Done in 0.38s  |  inertia=1.1457e+07
